## GPU-Accelerated Deep Learning Script

This notebook demonstrates a simple Convolutional Neural Network (CNN) for image classification using TensorFlow/Keras, designed to leverage GPU acceleration for faster training.

### 1. Check for GPU Availability

It's crucial to verify that a GPU is available and recognized by TensorFlow. Colab typically provides free GPU access; ensure your runtime type is set to GPU (Runtime > Change runtime type > GPU).

In [1]:
import tensorflow as tf

print("TensorFlow Version:", tf.__version__)

gpu_available = tf.config.list_physical_devices('GPU')

if gpu_available:
    print(f"GPU(s) available: {len(gpu_available)}")
    for gpu in gpu_available:
        print(f"  - {gpu}")
    tf.config.experimental.set_memory_growth(gpu_available[0], True)
    print("Memory growth for GPU set to True.")
else:
    print("No GPU available. TensorFlow will use CPU.")


TensorFlow Version: 2.20.0
GPU(s) available: 1
  - PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')
Memory growth for GPU set to True.


### 2. Load and Preprocess Dataset

We'll use the MNIST dataset, a classic dataset of handwritten digits, which is small and easy to load for demonstration purposes. The images will be normalized and reshaped for the CNN.

In [2]:
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

# Load MNIST dataset
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

# Reshape images to add a channel dimension (needed for CNNs)
# MNIST images are 28x28 grayscale, so we need to add a 1 for the channel.
train_images = train_images.reshape((60000, 28, 28, 1)).astype('float32') / 255
test_images = test_images.reshape((10000, 28, 28, 1)).astype('float32') / 255

# Convert labels to one-hot encoding
train_labels = to_categorical(train_labels)
test_labels = to_categorical(test_labels)

print(f"Training images shape: {train_images.shape}")
print(f"Training labels shape: {train_labels.shape}")
print(f"Test images shape: {test_images.shape}")
print(f"Test labels shape: {test_labels.shape}")


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Training images shape: (60000, 28, 28, 1)
Training labels shape: (60000, 10)
Test images shape: (10000, 28, 28, 1)
Test labels shape: (10000, 10)


### 3. Build the Convolutional Neural Network (CNN) Model

Here, we define a simple CNN architecture. It includes convolutional layers for feature extraction, pooling layers for dimensionality reduction, and dense layers for classification.

In [3]:
from tensorflow.keras import layers, models

model = models.Sequential()
model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation='relu'))
model.add(layers.Flatten())
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dense(10, activation='softmax'))

model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 3, 3, 64)       │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 576)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 93,322 (364.54 KB)

 Trainable params: 93,322 (364.54 KB)

 Non-trainable params: 0 (0.00 B)

### 4. Compile and Train the Model

We compile the model by defining the optimizer, loss function, and metrics. Then, we train the model using the preprocessed training data. The GPU will significantly speed up this process compared to CPU-only training.

In [4]:
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

print("Starting model training...")

history = model.fit(train_images, train_labels, epochs=5, batch_size=64, validation_split=0.2)

print("Model training complete.")


Starting model training...
Epoch 1/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - accuracy: 0.9329 - loss: 0.2187 - val_accuracy: 0.9788 - val_loss: 0.0751
Epoch 2/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9830 - loss: 0.0556 - val_accuracy: 0.9828 - val_loss: 0.0562
Epoch 3/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9888 - loss: 0.0372 - val_accuracy: 0.9865 - val_loss: 0.0453
Epoch 4/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9911 - loss: 0.0290 - val_accuracy: 0.9881 - val_loss: 0.0398
Epoch 5/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9930 - loss: 0.0221 - val_accuracy: 0.9872 - val_loss: 0.0523
Model training complete.


### 5. Evaluate the Model

Finally, we evaluate the trained model on the test dataset to see its performance on unseen data.

In [5]:
test_loss, test_acc = model.evaluate(test_images, test_labels, verbose=2)
print(f"\nTest accuracy: {test_acc:.4f}")


313/313 - 1s - 5ms/step - accuracy: 0.9883 - loss: 0.0410

Test accuracy: 0.9883


## Larger Training Example: CIFAR-10 Dataset

Let's work with a more challenging image classification task: the CIFAR-10 dataset. This dataset consists of 60,000 32x32 color images in 10 classes, with 6,000 images per class. There are 50,000 training images and 10,000 test images.

### 1. Load and Preprocess CIFAR-10 Dataset

We'll load the CIFAR-10 dataset, normalize the pixel values, and convert the labels to one-hot encoding, similar to how we handled MNIST, but accounting for the 3 color channels.

In [ ]:
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical

# Load CIFAR-10 dataset
(cifar_train_images, cifar_train_labels), (cifar_test_images, cifar_test_labels) = cifar10.load_data()

# Normalize pixel values to be between 0 and 1
cifar_train_images = cifar_train_images.astype('float32') / 255
cifar_test_images = cifar_test_images.astype('float32') / 255

# Convert labels to one-hot encoding
cifar_train_labels = to_categorical(cifar_train_labels, 10)
cifar_test_labels = to_categorical(cifar_test_labels, 10)

print(f"CIFAR-10 Training images shape: {cifar_train_images.shape}")
print(f"CIFAR-10 Training labels shape: {cifar_train_labels.shape}")
print(f"CIFAR-10 Test images shape: {cifar_test_images.shape}")
print(f"CIFAR-10 Test labels shape: {cifar_test_labels.shape}")


   778240/170498071 ━━━━━━━━━━━━━━━━━━━━ 1:17:23 27us/step

### 2. Build a CNN Model for CIFAR-10

For CIFAR-10, we'll build a slightly deeper CNN model with more convolutional layers and different filter sizes to better capture features from color images.

In [ ]:
from tensorflow.keras import layers, models

cifar_model = models.Sequential()
cifar_model.add(layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)))
cifar_model.add(layers.BatchNormalization())
cifar_model.add(layers.Conv2D(32, (3, 3), activation='relu', padding='same'))
cifar_model.add(layers.BatchNormalization())
cifar_model.add(layers.MaxPooling2D((2, 2)))
cifar_model.add(layers.Dropout(0.25))

cifar_model.add(layers.Conv2D(64, (3, 3), activation='relu', padding='same'))
cifar_model.add(layers.BatchNormalization())
cifar_model.add(layers.Conv2D(64, (3, 3), activation='relu', padding='same'))
cifar_model.add(layers.BatchNormalization())
cifar_model.add(layers.MaxPooling2D((2, 2)))
cifar_model.add(layers.Dropout(0.25))

cifar_model.add(layers.Conv2D(128, (3, 3), activation='relu', padding='same'))
cifar_model.add(layers.BatchNormalization())
cifar_model.add(layers.Conv2D(128, (3, 3), activation='relu', padding='same'))
cifar_model.add(layers.BatchNormalization())
cifar_model.add(layers.MaxPooling2D((2, 2)))
cifar_model.add(layers.Dropout(0.25))

cifar_model.add(layers.Flatten())
cifar_model.add(layers.Dense(512, activation='relu'))
cifar_model.add(layers.BatchNormalization())
cifar_model.add(layers.Dropout(0.5))
cifar_model.add(layers.Dense(10, activation='softmax'))

cifar_model.summary()


### 3. Compile and Train the CIFAR-10 Model

We'll compile the new model and train it using the CIFAR-10 data. Due to the increased complexity of the dataset and model, training will take longer and benefit significantly from the GPU.

In [ ]:
cifar_model.compile(optimizer='adam',
                    loss='categorical_crossentropy',
                    metrics=['accuracy'])

print("Starting CIFAR-10 model training...")

cifar_history = cifar_model.fit(cifar_train_images, cifar_train_labels, epochs=20, batch_size=128, validation_split=0.1)

print("CIFAR-10 Model training complete.")


### 4. Evaluate the CIFAR-10 Model

Finally, we evaluate the CIFAR-10 model on its test dataset.

In [ ]:
cifar_test_loss, cifar_test_acc = cifar_model.evaluate(cifar_test_images, cifar_test_labels, verbose=2)
print(f"\nCIFAR-10 Test accuracy: {cifar_test_acc:.4f}")
